# Act 1 — Run one server

**The problem:** you have an application and you need a computer, reachable from the internet, to run it on. Nothing elaborate yet — one machine, your code on it, a URL you can open in a browser.

That machine is an EC2 instance. In this act we make only the decisions needed to get there. Every option has real depth, but we defer all of it — the goal is a running, reachable server you can point at.

## What EC2 is

**EC2 (Elastic Compute Cloud)** is AWS's virtual server service — the most fundamental compute primitive in AWS. Almost every workload on AWS runs on EC2 in some form, either directly or as the engine behind a managed service.

An **instance** is a virtual machine: AWS chooses physical hardware, runs a hypervisor on it, and gives you a slice. You pick the OS, CPU, memory, storage, and networking; AWS handles the hardware underneath.

## The minimum to launch a working web server

To get from nothing to a server you can open in a browser, you make six decisions. Each has depth — we take all of that in Act 2. For now, the smallest viable choice for each:

| Decision | Happy-path choice | One-line reason |
|---|---|---|
| **AMI** (the disk image) | Amazon Linux 2023 | A maintained, free Linux to boot from |
| **Instance type** (size) | `t3.micro` | Cheap, free-tier eligible, fine for a demo |
| **Key pair** | one you create | Lets you SSH in if you need to |
| **Network** | default VPC, public IP on | Gives the instance a routable address |
| **Security group** (firewall) | allow `80` (HTTP) and `22` (SSH) | So a browser and your SSH client can reach it |
| **User data** (boot script) | install + start a web server | Turns a bare OS into a running app on first boot |

The boot script in the launch call below installs Apache and drops a one-line page. When the instance finishes booting, its public IP serves that page — that is your "it works" moment.

In [ ]:
import boto3

ec2 = boto3.client("ec2", region_name="us-east-1")

user_data = """#!/bin/bash
yum update -y
yum install -y httpd
systemctl enable --now httpd
echo "<h1>Hello from EC2</h1>" > /var/www/html/index.html
"""

response = ec2.run_instances(
    ImageId="ami-0c02fb55956c7d316",         # Amazon Linux 2 in us-east-1
    InstanceType="t3.micro",
    MinCount=1,
    MaxCount=1,
    KeyName="my-key-pair",
    SecurityGroupIds=["sg-xxxxxxxx"],
    IamInstanceProfile={"Name": "WebServerInstanceProfile"},
    UserData=user_data,
    MetadataOptions={"HttpTokens": "required"},   # enforce IMDSv2
    TagSpecifications=[{
        "ResourceType": "instance",
        "Tags": [{"Key": "Name", "Value": "demo-web"}],
    }],
)
print(response["Instances"][0]["InstanceId"])

You now have a running, internet-reachable server with your app on it. That is the whole of "compute" at its simplest: one instance, one image, one boot script.

Everything that follows is refinement. **Act 2** asks whether each of those six choices was the *right* one. **Acts 3 and 4** ask what happens when one server stops being enough.

# Act 2 — Make those choices well

You have a running instance, but you made every decision on autopilot: a default size, on-demand pricing, an off-the-shelf image, a script that ran on boot. Each was a fork in the road.

This act revisits the same decisions with judgment. Nothing here is new machinery — it is the *why* behind the fields you just filled in.

## What You Configure at Launch

| Setting | What it sets |
|---|---|
| **AMI** | OS image — Amazon Linux, Ubuntu, Windows, or a custom-built image |
| **Instance type** | CPU + memory profile (e.g. `m7g.large`) |
| **Key pair** | SSH key for Linux, RDP password decryption for Windows |
| **Network** | VPC, subnet, public IP toggle |
| **Security group** | Instance-level firewall rules |
| **Storage** | Root EBS volume + optional extra volumes |
| **IAM role** | Permissions via an instance profile — no long-lived keys on disk |
| **User data** | Shell script run once on first boot |

## Instance Families

Instance types group by workload shape. Pick the family first, then the size.

| Family | Optimised for | Examples | Typical workloads |
|---|---|---|---|
| **General Purpose** | Balanced CPU/memory/network | `t3`, `t4g`, `m7g`, `m6i` | Web servers, small DBs, dev environments |
| **Compute Optimised** | High CPU per dollar | `c7g`, `c6i` | Batch jobs, gaming servers, ML inference, HPC |
| **Memory Optimised** | High RAM per core | `r7g`, `x2idn`, `u-*` | In-memory DBs, Redis, SAP HANA |
| **Storage Optimised** | Local NVMe throughput | `i4i`, `d3`, `h1` | NoSQL, data warehouses, distributed file systems |
| **Accelerated Computing** | GPUs and custom chips | `p4`, `g5`, `trn1`, `inf2` | ML training/inference, video, scientific computing |
| **HPC Optimised** | Tightly coupled HPC | `hpc7g` | Large simulations, CFD, weather |

## Instance Naming

The full name encodes the workload, generation, capability suffixes, and size:

`m7g.xlarge`
- `m` — family (general purpose)
- `7` — generation
- `g` — Graviton (AWS ARM processor); other suffixes include `i` (Intel), `a` (AMD), `n` (network-optimised), `d` (local NVMe)
- `xlarge` — size (vCPU + RAM step)

Sizes scale roughly linearly with cost: `large` → `xlarge` → `2xlarge` → `4xlarge` doubles vCPU and memory each step.

**Graviton (`g`) instances** are typically ~20% cheaper and ~40% better performance-per-dollar than equivalent x86 instances — the default first choice when the workload runs on ARM.

## T-Series Burstable Instances

The `t3` / `t4g` families run at a **baseline CPU percentage** and earn CPU credits when running below baseline. Bursts above baseline spend credits.

| Mode | Behaviour when credits run out |
|---|---|
| **Standard** | CPU is capped at baseline |
| **Unlimited** (default for `t3`/`t4g`) | Continues to burst at extra cost |

Use t-series for workloads with idle stretches and occasional spikes — low-traffic web servers, dev boxes, internal tools. **Don't** use them for sustained-CPU workloads (you'll either throttle or pay unlimited-mode overage).

## Purchasing Options

The single most important EC2 cost lever. Each option trades commitment for discount.

| Option | Commitment | Discount | When to use |
|---|---|---|---|
| **On-Demand** | None | 0% | Unpredictable or short-term workloads |
| **Reserved Instances (1yr)** | 1 year | ~40% | Steady 24/7 workloads |
| **Reserved Instances (3yr)** | 3 years | up to ~72% | Long-term stable workloads |
| **Savings Plans** | 1 or 3 years | up to ~72% | Flexible commit across EC2/Lambda/Fargate |
| **Spot** | None | up to ~90% | Fault-tolerant, interruption-safe workloads |
| **Dedicated Host** | On-demand or reserved | — | BYOL, socket-level licensing, compliance |
| **Dedicated Instance** | On-demand | small premium | Physical isolation without BYOL |
| **Capacity Reservation** | None | 0% | Guaranteed capacity in a specific AZ |

## Reserved Instances — Anatomy

A Reserved Instance is a billing construct, not a separate kind of instance. You commit to a specific instance shape and AWS applies a discount to any matching On-Demand instance running in your account.

**Payment options** (more upfront = bigger discount):
- All Upfront — entire term paid now
- Partial Upfront — some now, rest monthly
- No Upfront — monthly only, smallest discount

**Scope:**
| Scope | Flexibility | Capacity reservation |
|---|---|---|
| **Regional** | Applies across any AZ in the region; size-flexible within the family (Linux only) | No |
| **Zonal** | Locked to a specific AZ and exact size | Yes |

**Convertible RIs** allow changing family, OS, tenancy mid-term in exchange for a smaller discount — the escape hatch when you don't know what you'll be running in two years.

## Savings Plans vs Reserved Instances

Savings Plans are the modern replacement for RIs in most cases. You commit to a steady spend rate (e.g. `$10/hr for 1 year`) rather than to specific instances.

| | Reserved Instance | Savings Plan |
|---|---|---|
| **Commit** | Specific instance family/region | Hourly dollar amount |
| **Flexibility** | Limited (within family) | High — auto-applies across instances |
| **Applies to** | EC2 only | EC2, Fargate, Lambda (Compute SP) |
| **Marketplace resale** | Yes (Standard RIs) | No |

**Compute Savings Plans** are the most flexible (any region, any family, EC2/Fargate/Lambda). **EC2 Instance Savings Plans** lock you to a family in a region but give a deeper discount.

> The general rule: prefer **Compute Savings Plans** as the default commitment instrument. Reach for RIs only when you specifically need a capacity reservation or marketplace resale.

## Spot Instances

Spot uses AWS's spare capacity. Discounts of 70–90% off On-Demand, but AWS can **reclaim** the instance with a **2-minute warning** when capacity is needed elsewhere.

**Mechanics:**
- Spot price varies smoothly over time (no more bidding, post-2017)
- An interruption fires the `instance-action` metadata, then EC2 stops or terminates the instance
- Spot Fleet / EC2 Fleet diversify across instance types and AZs to reduce simultaneous-interruption risk

**Good fits:**
- Batch processing, CI/CD runners, big data (EMR/Spark)
- Stateless web workers behind a load balancer
- Fault-tolerant ML training with checkpointing

**Bad fits:**
- Databases or anything with non-checkpointable state
- Single-AZ dependencies that can't tolerate eviction
- Workloads with strict SLAs

**Spot Blocks** (reserved spot capacity for 1–6 hours) are no longer available to new customers — diversification is now the only recommended interruption-tolerance pattern.

## Amazon Machine Images (AMIs)

An **AMI** is the template an instance launches from. It bundles the OS, pre-installed software, and any baked-in config. The AMI itself is metadata; the bytes live in EBS snapshots that AWS manages on your behalf.

**Sources:**
| Source | Description |
|---|---|
| **AWS-provided** | Amazon Linux, Ubuntu, Windows Server, etc. — patched and maintained by AWS |
| **Marketplace** | Vendor AMIs with software pre-installed and licensing included |
| **Community** | Public AMIs shared by other accounts |
| **Custom** | AMIs you build yourself — your golden image |

**AMIs are region-scoped.** The same logical AMI has a different ID in every region. To use one elsewhere, you `CopyImage` it (which creates new snapshots in the destination region).

## AMI Lifecycle

```text
running instance → CreateImage → AMI + snapshots
                                    ├── RunInstances → new instances
                                    ├── CopyImage    → AMI in another region
                                    ├── share        → other accounts can launch
                                    └── DeregisterImage → AMI removed (snapshots NOT deleted)
```

**Things people miss:**
- `DeregisterImage` removes the AMI but the **snapshots remain** and keep billing — delete them explicitly.
- Sharing an **encrypted** AMI requires also sharing the **KMS key** used to encrypt the snapshots; otherwise the recipient can't launch.
- `NoReboot=False` (the default for `CreateImage`) gives a clean filesystem snapshot. `NoReboot=True` is faster but risks an inconsistent image.

## Baked vs Bootstrapped

| Approach | Launch speed | Flexibility | Use when |
|---|---|---|---|
| **Fully bootstrapped** | Slowest — full setup in User Data | Highest — config changes need no rebuild | Rapid iteration, dev/test |
| **Hybrid** | Medium — base AMI has dependencies, app pulled at launch | Balanced | Most production fleets |
| **Fully baked** | Fastest — app and config are in the AMI | Lowest — every change is a rebuild | Auto Scaling at scale, immutable infra |

**EC2 Image Builder** is the AWS-native pipeline for producing golden AMIs: pick a base image, layer build components (packages, scripts), run test components, then distribute to target regions and accounts. Triggers can be on a schedule or whenever the base AMI updates.

## User Data

A script (or `cloud-init` directive) supplied at launch and run **once**, as root, on first boot. Used to install packages, pull code, configure the OS, or register the instance with a discovery service.

```bash
#!/bin/bash
yum update -y
yum install -y httpd
systemctl enable --now httpd
echo "<h1>Hello from EC2</h1>" > /var/www/html/index.html
```

- Maximum size: **16 KB** (Base64-encoded on the API). Larger bootstrap logic should be fetched from S3 by a short user-data stub.
- Output goes to `/var/log/cloud-init-output.log` — first place to look when an instance comes up "running" but the app doesn't respond.
- To re-run user data on every boot, prefix the script with the cloud-init mime header `--//` or rewrite `/etc/cloud/cloud.cfg`. By default it runs **only once**.

## Security Groups

A security group is a stateful, allow-only firewall attached to an ENI (and therefore to the instance it belongs to).

| Property | Detail |
|---|---|
| **Stateful** | If inbound is allowed, the response is allowed back automatically (and vice versa) |
| **Allow rules only** | No `Deny` — for denies, use a NACL at the subnet level |
| **Default inbound** | Deny all |
| **Default outbound** | Allow all |
| **Sources** | CIDR block, single IP, or **another security group ID** |
| **Composition** | An ENI can have multiple SGs; rules are unioned |

**SG-to-SG references** are the cleanest pattern for tiered architectures: the database SG allows port 5432 from the app SG, not from a hard-coded IP range. The membership relationship updates automatically as instances come and go.

> SGs are **not** a substitute for NACLs. SGs are stateful and per-instance; NACLs are stateless and per-subnet. They complement each other.

## Instance Metadata Service (IMDS)

Code on the instance can query a link-local endpoint at `169.254.169.254` to discover facts about itself: instance ID, AZ, IAM role credentials, attached tags, user data, public/private IPs.

| Version | How it works | Security |
|---|---|---|
| **IMDSv1** | Simple `GET` — no authentication | Vulnerable to SSRF: a server-side bug that fetches an attacker-controlled URL can leak role credentials |
| **IMDSv2** | `PUT` first to get a session token, then `GET` with the token in a header | SSRF-resistant — token is bound to the instance |

Newer instance launches default to **IMDSv2-required**. Treat IMDSv1 as deprecated — it's the single biggest cause of historic credential-theft incidents on EC2.

## Public, Private, and Elastic IPs

| | Public IP | Private IP | Elastic IP |
|---|---|---|---|
| **Scope** | Internet-routable | VPC-only | Internet-routable, account-owned |
| **Lifecycle** | Auto-assigned at launch (if enabled); **changes on stop/start** | Permanent for the lifetime of the ENI | Allocated separately; survives stop/start |
| **Cost** | Now charged hourly for all public IPv4 addresses (since Feb 2024) | Free | Free while attached to a **running** instance; charged when idle |
| **Use it for** | Throwaway, short-lived instances | All intra-VPC traffic | Stable inbound endpoint, IP whitelisting, DNS pin |

**IPv6** in AWS is always public-routable (there's no NAT for IPv6) — to keep IPv6 instances unreachable from the internet, use an **Egress-Only Internet Gateway**.

## Tenancy Options

| Tenancy | Hardware sharing | Visible to you | BYOL / socket licensing |
|---|---|---|---|
| **Shared** (default) | Shared with other customers | No | No |
| **Dedicated Instance** | Hardware reserved for your account | No | No |
| **Dedicated Host** | Specific physical server | Yes — sockets, cores, host ID | Yes |
| **Bare Metal** | Specific server, no hypervisor | Direct hardware access | Yes |

Use a **Dedicated Host** when a vendor's licence is tied to physical sockets (e.g. some Oracle, Windows Server licences) or when an audit requires host-level visibility. Otherwise default to shared tenancy — it's cheapest and gives the same isolation guarantees at the hypervisor level.

## ENIs, ENA, and EFA

| | What it is | When you need it |
|---|---|---|
| **ENI** (Elastic Network Interface) | A virtual NIC. Every instance has at least one (`eth0`). Carries the private IP, MAC, security groups, and optional public/Elastic IPs. | Always — it's the fundamental network attachment |
| **ENA** (Elastic Network Adapter) | The high-throughput driver enabling up to **100+ Gbps** on supported instances. | Default on Nitro instances — no setup required |
| **EFA** (Elastic Fabric Adapter) | A specialised ENI that bypasses the OS kernel for HPC/ML traffic, exposing the **OS-bypass libfabric** interface for MPI and NCCL workloads. | Tightly coupled HPC, distributed ML training on `p4d`, `p5`, `hpc*` instances |

Multiple ENIs on one instance enable patterns like dual-homed appliances (management plane + data plane on separate subnets) and floating IPs that move between instances on failover.

## Placement Groups

By default AWS spreads instances across hardware to minimise correlated failure. Placement groups override that default — packing tighter for performance, or further apart for resilience.

### Cluster Placement Group

All instances on the **same rack** (same AZ, same low-latency network spine).

- **Latency:** lowest possible between instances
- **Bandwidth:** up to 10 Gbps single-flow (vs 5 Gbps cross-AZ)
- **AZ:** single AZ only
- **Risk:** rack-level failure takes the whole group down
- **Use for:** tightly coupled HPC, distributed training, big data with heavy shuffle

### Spread Placement Group

Each instance on a **different rack** — distinct power, networking, and host hardware.

- **AZ:** can span multiple AZs
- **Limit:** **max 7 instances per AZ** per placement group
- **Use for:** small fleets of critical instances where you can't tolerate two of them failing together — HA control plane nodes, primary + standby DB pairs, ZooKeeper ensembles

### Partition Placement Group

Instances grouped into **partitions**; each partition has its own racks, with no shared hardware between partitions.

- **Scale:** up to 7 partitions per AZ, hundreds of instances total
- **Topology-aware:** instances can read their partition number from IMDS
- **Use for:** distributed systems that already understand rack topology — HDFS, Cassandra, Kafka, HBase

Choosing between the three:

| Need | Use |
|---|---|
| Lowest inter-node latency, small fleet | Cluster |
| Maximum isolation, small fleet | Spread |
| Large topology-aware distributed system | Partition |

## The Nitro System

The platform underlying all modern EC2 instances (most 5th-gen and all later). Nitro **offloads** virtualisation work — networking, storage, security — from the host CPU onto dedicated Nitro cards, and runs a stripped-down hypervisor on the host.

| Capability | What you get |
|---|---|
| **Near bare-metal performance** | Minimal hypervisor overhead — most host CPU and RAM go to your instance |
| **Bare metal instances** | No hypervisor at all (`*.metal` sizes) |
| **Hardware EBS encryption** | Encryption performed on the Nitro card — no CPU cost |
| **Higher network ceilings** | Up to 100–200 Gbps depending on instance |
| **Security boundary** | Host is immutable; AWS operators have no path into guest memory |

**Bare metal** is rarely chosen for application workloads. The usual reasons: running a nested hypervisor (e.g. VMware Cloud on AWS), licensing that requires a physical socket, or measuring CPU/RAM behaviour without the hypervisor in the way.

## Hibernate, Stop, Terminate

| Action | RAM | Root EBS | Instance Store | Billing |
|---|---|---|---|---|
| **Stop** | Lost | Retained | Lost | Compute stops; EBS still billed |
| **Hibernate** | **Saved to encrypted root EBS** | Retained | Lost | Compute stops; EBS still billed |
| **Terminate** | Lost | Deleted (unless overridden) | Lost | Stops entirely |

**Hibernate** writes RAM to the root volume and stops the instance. On resume, RAM is restored and the OS doesn't boot — the process tree picks up exactly where it left off.

Requirements: encrypted root volume large enough to hold RAM, RAM under 150 GB, supported instance type, max 60-day hibernation. Use for slow-to-warm services (heavy in-memory caches, JIT-warmed runtimes) where a normal stop would force a long re-warming on every restart.

# Act 3 — One server isn't enough

Two things go wrong with a single instance. It dies at 3am and your app is down until someone notices. And you can't ship a new version without taking the whole site offline.

The fix is to run **more than one** instance — which immediately raises a new question: if there are several instances, how does a user reach *the app* rather than picking one by IP address? That is the job of a load balancer.

## Elastic Load Balancing

A load balancer accepts incoming connections at a stable endpoint and distributes them across a fleet of targets. The value it adds:

- **Health checks** — unhealthy targets stop receiving traffic
- **Single DNS endpoint** — clients don't track individual instances
- **Connection draining** — graceful removal of instances mid-deploy
- **TLS termination** — certificate lives on the LB, not on every instance
- **Stickiness** — same client returns to the same target (when needed)

## The Four ELB Types

| | ALB | NLB | GWLB | CLB *(legacy)* |
|---|---|---|---|---|
| **Layer** | 7 (HTTP/HTTPS/gRPC) | 4 (TCP/UDP/TLS) | 3 (IP, GENEVE) | 4/7 |
| **Routing** | Content-based (path, host, header, query, source IP) | Connection-based | Transparent pass-through | Round-robin |
| **Static IP** | No (DNS only) | Yes — one per AZ | Yes | No |
| **TLS** | Terminates | Terminate or pass-through | Pass-through | Terminates |
| **Typical workload** | Web apps, APIs, microservices | Real-time, gaming, financial, IP whitelisting | Inline firewalls, IDS, packet inspection | Legacy migrations only |

CLB exists only for legacy reasons. New designs use ALB for HTTP workloads, NLB for everything else.

## Application Load Balancer (ALB)

ALB understands HTTP/HTTPS. It terminates the connection, parses the request, and routes based on its content.

**Object model:**

```text
Listener (port + protocol)
  └── Rules (evaluated in priority order)
        ├── Conditions: path, host, header, query string, source IP, HTTP method
        └── Action: forward / weighted forward / redirect / fixed response / authenticate
              └── Target Group → Targets (EC2 / IP / Lambda / another ALB)
```

A **target group** is the unit of routing: it has its own health check, its own protocol/port, and one of three target types — `instance` (auto-registered by ASG), `ip` (for on-prem or peered targets), or `lambda` (synchronous Lambda invocation).

### ALB Listener Actions

| Action | What it does |
|---|---|
| **Forward** | Send the request to a target group |
| **Weighted forward** | Split traffic across target groups by weight — canary deploys, blue/green, A/B tests |
| **Redirect** | Return a 301/302 to a different URL — most commonly HTTP → HTTPS |
| **Fixed response** | Return a static HTTP response without forwarding — maintenance pages |
| **Authenticate-Cognito / OIDC** | Force the user through an identity provider before the request reaches your app |

ALB also natively supports **gRPC**, **WebSockets**, **HTTP/2**, server-sent events, and **WAF** attachment for Layer 7 rules.

**Stickiness** uses an LB-generated cookie (`AWSALB`) and is configured per target group, with a duration up to 7 days.

## Network Load Balancer (NLB)

NLB operates at Layer 4. It moves packets — it does not parse them — which makes it both fast and protocol-agnostic.

| Property | Detail |
|---|---|
| **Static IPs** | One per AZ, allocated by AWS or assigned as an Elastic IP |
| **Source IP** | **Preserved** — the target sees the client's real IP (ALB rewrites it via `X-Forwarded-For`) |
| **Throughput** | Millions of requests per second; single-digit-ms latency |
| **TLS** | Can terminate (with an ACM cert) or pass through to the target |
| **Protocols** | TCP, UDP, TCP_UDP, TLS — **only LB type that supports UDP** |

Reach for NLB when:
- You need a **static IP** for firewall whitelisting
- Traffic isn't HTTP (gaming, IoT, DNS, custom protocols)
- You need to see the client's real source IP without `X-Forwarded-For`
- Latency at the LB matters more than Layer-7 features

## Gateway Load Balancer (GWLB)

GWLB sits inline as a transparent traffic-bump for third-party network appliances — next-gen firewalls, IDS/IPS, deep packet inspection.

```
Client traffic ─► GWLB ─► appliance fleet (firewalls/IDS) ─► GWLB ─► destination
```

It uses the **GENEVE protocol on port 6081** to encapsulate the original packet, hand it to an appliance, then deliver the (possibly modified) packet to its real destination. The source and destination workloads don't know the appliance is in the path.

GWLB is niche — most architectures never need it. Recognise it when you see "deploy a fleet of third-party firewalls scalably and transparently" in a design.

## Target Groups, Health Checks, Draining

A target group is reused across listeners and rules; multiple ALBs/NLBs can point to the same target group.

**Health check:**
| Setting | Default |
|---|---|
| Protocol + path (e.g. HTTP `/health`) | configured |
| Healthy threshold (consecutive passes to mark healthy) | 5 (ALB) / 3 (NLB) |
| Unhealthy threshold | 2 (ALB) / 3 (NLB) |
| Interval | 30 s |
| Timeout | 5 s |

**Deregistration delay** (a.k.a. connection draining): when a target is removed from a target group, the LB stops sending new requests but waits up to this duration for in-flight requests to complete. Default 300 s. Lower it (e.g. 30 s) for stateless services; keep it higher for long-lived or websocket connections.

**Cross-zone load balancing:**
| LB | Default | Charge if enabled |
|---|---|---|
| ALB | **On** | Free |
| NLB | **Off** | Cross-AZ data transfer charges apply |
| GWLB | **Off** | Cross-AZ data transfer charges apply |

# Act 4 — Traffic varies

Now you run a fleet behind a load balancer. But the fleet size is still something you set by hand. Traffic triples at lunch and the site slows while you're asleep; it goes quiet overnight and you pay for idle servers.

You want something that watches load and adds or removes instances for you, keeping the fleet registered with the load balancer as it changes. That is an **Auto Scaling Group**.

## Auto Scaling Groups (ASG)

An ASG manages a fleet of EC2 instances: it keeps a target count running, replaces failed instances, and scales the count up or down based on policies you attach.

**Core capacity settings:**
| Setting | Meaning |
|---|---|
| **Minimum** | Floor — ASG never goes below this count |
| **Desired** | Target the ASG actively maintains |
| **Maximum** | Ceiling — scaling will not exceed this |

The ASG distributes instances across the subnets you give it (one per AZ for HA). It attaches to one or more **target groups**, so newly launched instances are auto-registered with the load balancer and terminated instances are auto-deregistered.

## Launch Templates

A **launch template** describes how an ASG (or a one-off `RunInstances` call) should configure a new instance: AMI, instance type(s), key pair, security groups, IAM role, user data, storage, network config, IMDS options.

| | Launch Template | Launch Configuration *(legacy)* |
|---|---|---|
| **Versioning** | Yes — multiple versions, `$Latest`, `$Default` | None |
| **Editing** | Create a new version | Immutable; must replace entirely |
| **Mixed instance types** | Yes — combine On-Demand + Spot, multiple types | No |
| **New EC2 features** | Supported | Frozen — many features unavailable |

Always use launch templates. AWS has effectively deprecated launch configurations.

## Scaling Policies

ASGs adjust capacity through one of several policy types, often combined.

### Target Tracking

Pick a metric and a target value; AWS automatically adds and removes instances to hold the metric at the target.

```
Keep average CPU at 50%
Keep RequestCountPerTarget at 1000
Keep average network-in at 1 GB/s
```

The simplest policy and the right default. AWS provisions a pair of CloudWatch alarms behind the scenes and handles cooldowns internally. Works for any metric that scales roughly linearly with capacity.

### Step Scaling

Define a CloudWatch alarm, then specify scaling steps — how many instances to add or remove based on how badly the alarm is breached.

```text
CPU 50–70%  → add 1
CPU 70–90%  → add 2
CPU > 90%   → add 4
```

Use step scaling when the relationship between metric and required capacity is non-linear, or when you want sharper response above a critical threshold than target tracking would give.

### Scheduled and Predictive Scaling

**Scheduled scaling** sets desired/min/max at specific times — for predictable workloads (business hours, end-of-month batches, weekend traffic drops).

```text
Mon–Fri 08:00 UTC  → min=10, desired=20
Mon–Fri 20:00 UTC  → min=2,  desired=5
```

**Predictive scaling** uses ML on the last two weeks of CloudWatch data to forecast load and **pre-scale** the ASG before demand arrives. Pair it with target tracking — predictive handles the forecastable bulk, target tracking absorbs the residual variance.

**Simple scaling** (one step, with a cooldown blocking further triggers) is the legacy version of step scaling. Don't use it for new designs.

## ASG Health Checks

| Type | What's checked | Catches |
|---|---|---|
| **EC2** (default) | EC2 status checks — instance is running, hypervisor healthy | Host failures, kernel hangs |
| **ELB** | Target group health check passes | App-level failures — process crashed, returning 5xx |
| **Custom** | Your own signal via `SetInstanceHealth` | Whatever you decide |

**Always enable ELB health checks on production ASGs.** Without them, an instance that's running but returning 500s on every request still counts as healthy, and the ASG will happily leave it serving traffic.

**Health check grace period** delays health checks for new instances by N seconds so they can boot and pass before being evaluated. Set this to a comfortable margin above the slowest cold start in your fleet.

## Instance Refresh

When the launch template updates (new AMI, new user data, new instance type), **Instance Refresh** rolls out the change across the fleet without manual intervention.

- **Minimum healthy percentage** — e.g. 90% — caps how many instances are out of service at once
- **Warm-up time** — how long before a new instance is treated as in-service for the percentage calculation
- **Checkpoints** — pause between batches for verification
- Can be cancelled or rolled back mid-way

The pattern that pairs with this is **immutable infrastructure**: don't patch live instances; bake a new AMI, point the launch template at it, run Instance Refresh.

## Lifecycle Hooks

Lifecycle hooks pause an instance at a transition state so you can run custom logic before it joins or leaves the fleet.

```text
Scale-out:   Pending → Pending:Wait → Pending:Proceed → InService
                            ↑
              Bootstrap, register with service discovery,
              run smoke tests → complete-lifecycle-action

Scale-in:    InService → Terminating:Wait → Terminating:Proceed → Terminated
                                ↑
              Drain connections, upload logs, deregister
              from external systems → complete-lifecycle-action
```

The instance waits up to the configured timeout (default 1 hour, max 48 hours). If the timeout expires without an explicit `complete-lifecycle-action`, the ASG either continues (`CONTINUE`) or terminates the instance (`ABANDON`) based on the hook's default result.

## Warm Pools

A warm pool keeps a buffer of **pre-initialised** instances in `Stopped`, `Running`, or `Hibernated` state, ready to be activated quickly on scale-out.

```
No warm pool:   scale-out trigger → launch → bootstrap (2–10 min) → InService
With warm pool: scale-out trigger → take from pool → InService (seconds)
```

| Pool state | Cost | Activation speed |
|---|---|---|
| **Stopped** | EBS storage only | Slower — full OS boot |
| **Hibernated** | EBS storage only | Fast — RAM restored from disk |
| **Running** | Full compute charge | Fastest |

Use warm pools when bootstrap time is the bottleneck for scale-out — heavy AMIs, large data downloads, JVM warm-up. Pair with lifecycle hooks to do the warm-up work once, **before** the instance enters the pool, not on every scale-out.

# Act 5 — The whole pattern in code

Everything in this module converges on one shape: a load balancer in front of an Auto Scaling Group of EC2 instances, scaling on a metric. Here it is end to end — the payoff for the previous four acts.

In [ ]:
# Create an ALB + target group + HTTPS listener with HTTP→HTTPS redirect
elbv2 = boto3.client("elbv2", region_name="us-east-1")

alb = elbv2.create_load_balancer(
    Name="demo-alb",
    Subnets=["subnet-aaa", "subnet-bbb"],
    SecurityGroups=["sg-xxxxxxxx"],
    Scheme="internet-facing",
    Type="application",
)["LoadBalancers"][0]

tg = elbv2.create_target_group(
    Name="demo-web-tg",
    Protocol="HTTP",
    Port=80,
    VpcId="vpc-xxxxxxxx",
    TargetType="instance",
    HealthCheckPath="/health",
    HealthCheckIntervalSeconds=30,
    HealthyThresholdCount=2,
    UnhealthyThresholdCount=3,
)["TargetGroups"][0]

elbv2.create_listener(
    LoadBalancerArn=alb["LoadBalancerArn"],
    Protocol="HTTPS", Port=443,
    Certificates=[{"CertificateArn": "arn:aws:acm:us-east-1:111111111111:certificate/abc"}],
    DefaultActions=[{"Type": "forward", "TargetGroupArn": tg["TargetGroupArn"]}],
)

elbv2.create_listener(
    LoadBalancerArn=alb["LoadBalancerArn"],
    Protocol="HTTP", Port=80,
    DefaultActions=[{
        "Type": "redirect",
        "RedirectConfig": {"Protocol": "HTTPS", "Port": "443", "StatusCode": "HTTP_301"},
    }],
)

In [ ]:
# Create an ASG attached to the target group, with a target-tracking policy
asg = boto3.client("autoscaling", region_name="us-east-1")

asg.create_auto_scaling_group(
    AutoScalingGroupName="demo-asg",
    LaunchTemplate={"LaunchTemplateName": "demo-lt", "Version": "$Latest"},
    MinSize=2, DesiredCapacity=2, MaxSize=10,
    TargetGroupARNs=[tg["TargetGroupArn"]],
    HealthCheckType="ELB",
    HealthCheckGracePeriod=120,
    VPCZoneIdentifier="subnet-aaa,subnet-bbb",
)

asg.put_scaling_policy(
    AutoScalingGroupName="demo-asg",
    PolicyName="cpu-50",
    PolicyType="TargetTrackingScaling",
    TargetTrackingConfiguration={
        "PredefinedMetricSpecification": {"PredefinedMetricType": "ASGAverageCPUUtilization"},
        "TargetValue": 50.0,
    },
)

That is the canonical web-tier pattern on AWS, in about forty lines:

1. **EC2** runs the code (via the launch template the ASG references).
2. **ELB** gives users one stable endpoint and health-checks the targets.
3. **ASG** keeps the fleet at the right size and re-registers instances as they come and go.

Read top to bottom, the three services you met separately are now one system — each solving the problem the previous one exposed.